# CSE5280 —Robot Arm Evacuation Simulation
**Florida Institute of Technology**
*Bryan Bermudez | Spring 2026*

## Table of Contents

1. [Overview & Introduction](#1-overview--introduction)
2. [Mathematical Background](#2-mathematical-background)
3. [Utility & Math Functions](#3-utility--math-functions)
4. [Ramp & 3D Geometry Functions](#4-ramp--3d-geometry-functions)
5. [Cost Terms — Surface Adherence & Full Cost](#5-cost-terms--surface-adherence--full-cost)
6. [Robot Arm — IK Framework](#6-robot-arm--ik-framework)
7. [Clustering & Prediction](#7-clustering--prediction)
8. [Robot-Interference Simulation](#8-robot-interference-simulation)
9. [Robot Video Export](#9-robot-video-export)
10. [Robot Discussion & Results](#10-robot-discussion--results)

---

## 1. Overview & Introduction

This notebook implements a multi-particle building evacuation simulation using **gradient descent on a scalar cost function**. N (default N = 25) agents are distributed across three floors of a building and each agent follows the negative gradient of a composite cost landscape to navigate toward one of two ground-floor exits.

Everything is done with costs with no hardcoded path dectection, exit dection, wall dectection, etc

The simulation is rendered using the vedo library.

---

## 2. Mathematical Background

### 2.1 Ramp Model

A ramp connects two floors via a continuously-inclined surface. It is defined by a 2D centerline segment $[A, B]$ and a capsule half-width $r$. For any point $P = (x, y)$ we project onto the centerline:

$$t = \frac{(P - A) \cdot v}{v \cdot v}, \qquad v = B - A \tag{4.4, 4.5}$$

The projection is clamped to restrict to the finite ramp segment:

$$u(x, y) = \text{clip}(t,\, 0,\, 1) \tag{4.6}$$

The ramp height interpolates linearly between $z_0$ at $A$ and $z_1$ at $B$:

$$z_{\text{ramp}}(x, y) = z_0 + (z_1 - z_0)\, u(x, y) \tag{4.7}$$

The ramp footprint is a capsule around the centerline:

$$\mathcal{R} = \{(x,y) \mid \text{dist}((x,y),\,[A,B]) \leq r\} \tag{4.9}$$

### 2.2 Cost Function

Each particle with position $\mathbf{p} = (x, y, z)$ minimizes the composite cost:

$$C = C_{\text{goal}} + C_{\text{walls}} + C_{\text{height}} + C_{\text{smoothness}} + C_{\text{repulsion}} \tag{5.6}$$

**Goal attraction — soft-min :** A hard $\min(C_1, C_2)$ :

$$C_{\text{goal}}(\mathbf{p}) = -\tau \log\!\left(\sum_{i=1}^{2} \exp\!\left(-\frac{C_i(\mathbf{p})}{\tau}\right)\right), \qquad C_i(\mathbf{p}) = \|\mathbf{p} - \mathbf{p}^{\text{exit}}_i\|^2 \tag{5.7, 5.9}$$

Temperature $\tau \in [0.5, 1.5]$: smaller $\tau$ sharpens exit selection, larger $\tau$ blends exits more evenly. We use $\tau = 1.0$.

**Wall penalty — quadratic band:**

$$C_{\text{walls}} = \sum_k w_k\, \phi(d_k(\mathbf{p})), \qquad \phi(d) = \tfrac{1}{2}(R - d)^2\; \mathbf{1}[d \leq R] \tag{5.13}$$

**Surface-adherence cost:**

$$C_{\text{height}} = w_h \bigl(z - z_{\text{surf}}(x,y)\bigr)^2 \tag{5.2}$$

$$z_{\text{surf}}(x,y) = (1 - g)\,z_{\text{floor}} + g\,z_{\text{ramp}}(x,y), \qquad g = \begin{cases}1 & (x,y) \in \mathcal{R} \\ 0 & \text{otherwise}\end{cases} \tag{5.3, 5.4}$$

**Smoothness:**

$$C_{\text{smooth}} = \|\mathbf{p}_{k+1} - \mathbf{p}_k\|^2 \tag{5.14}$$

**Agent repulsion:**

$$C_{\text{repulsion}} = \sum_{j \neq i} \phi(\|\mathbf{p}_i - \mathbf{p}_j\|) \tag{5.15}$$

### 2.3 Optimization

Gradient descent with momentum smoothing and step-size capping:

$$\mathbf{v}_{k+1} = \mu\,\mathbf{v}_k - \alpha\,\nabla C(\mathbf{p}_k)$$
$$\mathbf{p}_{k+1} = \mathbf{p}_k + \text{clip}(\mathbf{v}_{k+1},\, -\Delta_{\max},\, \Delta_{\max})$$

Gradients are computed numerically via central finite differences. An agent is evacuated once its $(x,y)$ is within tolerance of either exit on the ground floor.

---

## 3. Utility & Math Functions

The cell below contains foundational geometry and cost helpers carried over from previous assignments.

- `point_to_segment_distance` — Euclidean distance from a point to a finite line segment (Eq. 4.4–4.6). Used in all wall penalty and ramp footprint checks.
- `wall_penalty_quad` — Quadratic-band wall repulsion $\phi(d) = \tfrac{1}{2}(R-d)^2$ for $d \leq R$ (Eq. 5.13).
- `goal_cost` — Squared-distance to a single exit (Eq. 5.7).
- `wall_cost` — Sums wall penalties over all wall segments (Eq. 5.13).
- `social_cost_quadratic` — Quadratic-band agent repulsion kernel (Eq. 5.15).

In [1]:
import numpy as np
import matplotlib.pyplot as plt


def point_to_segment_distance(x, a, b):
    """
    Distance from point x to line segment [a, b]
    """
    v = b - a       # direction vector of segment, used in eq 4.4: v = B - A
    w = x - a       # vector from segment start to point, used in eq 4.5: (P - A)
    vv = np.dot(v, v)   # v · v, the denominator in eq 4.5

    if vv == 0.0:
        return np.linalg.norm(x - a)

    # eq 4.5: t = (P - A) · v / (v · v)  — signed fractional distance along infinite line
    t = np.dot(w, v) / vv
    # eq 4.6: u(x,y) = clip(t, 0, 1)  — clamped to finite segment
    t = np.clip(t, 0.0, 1.0)
    q = a + t * v   # closest point on segment to x
    return np.linalg.norm(x - q)   # distance from x to closest point on segment


# -------------------------------
# Wall penalty (quadratic band)
# -------------------------------
def wall_penalty_quad(x, a, b, R):
    # eq 5.13: Cwalls = sum_k w_k * phi(d_k(p))
    # this is the quadratic band variant: phi(d) = 0.5 * (R - d)^2 when d <= R
    d = point_to_segment_distance(x, a, b)
    if d <= R:
        return 0.5 * (R - d)**2
    else:
        return 0.0


# bassically calculates the distance from the x to the goal the higher it is the worse it is
def goal_cost(x, goal):
    # the distance formula basically, halfing to make it not run at mach 10 towards the goal might change it
    # eq 5.7: C_i(p) = ||p - p_exit_i||^2
    return np.sum((x - goal)**2)


# x is the current pos, walls is walls, penelaty is the selected type, weights is how strong the repulsion is
# r is the max distance before it has no wieght
def wall_cost(x, walls, penalty, weight, r):
    total = 0.0

    # goes through all the walls
    # eq 5.13: Cwalls = sum_k w_k * phi(d_k(p))  — summing penalty over all wall segments k
    for a, b in walls:
        if penalty == "quad":
            total += wall_penalty_quad(x, a, b, r)
        elif penalty == "log":
            total += wall_penalty_log(x, a, b, r)
        elif penalty == "exp":
            total += wall_penalty_exp(x, a, b, r)

    return total * weight   # w_k applied here as a uniform weight across all walls


def total_cost(x, goal, walls, penalty, weight, r):
    # eq 5.6: C = Cgoal + Cwalls + Cheight + Csmoothness + Cagent-repulsion
    # (this is the 2D version — height and smoothness terms added in 3D extension)
    c_goal = goal_cost(x, goal)
    c_walls = wall_cost(x, walls, penalty, weight, r)
    return c_goal + c_walls


# meant to be soft kinda like normal quadratic
def social_cost_quadratic(xi, xj, R=1.0, k_social=5.0):
    # eq 5.15: Crepulsion = sum_{i!=j} phi(||pi - pj||)
    # phi here is the quadratic band: 0.5 * k * (R - d)^2 when d <= R
    dij = np.linalg.norm(xi - xj)  # ||pi - pj||

    if dij <= R:
        return 0.5 * k_social * (R - dij)**2
    else:
        return 0.0


# calculates the graident of all the particples nearby the particle that is called upon
def gradient_social_numerical(xi, xj, social_cost_func, epsilon=1e-5, **params):
    # numerical gradient via central finite differences
    # approximates: grad[dim] = (C(xi + eps) - C(xi - eps)) / (2 * eps)
    # extend range(2) to range(3) for 3D
    grad = np.zeros(2)

    for dim in range(2):
        xi_plus = xi.copy()
        xi_minus = xi.copy()
        xi_plus[dim] += epsilon
        xi_minus[dim] -= epsilon

        cost_plus = social_cost_func(xi_plus, xj, **params)
        cost_minus = social_cost_func(xi_minus, xj, **params)

        grad[dim] = (cost_plus - cost_minus) / (2 * epsilon)

    return grad

## 4. Ramp & 3D Geometry Functions

This section extends the 2D geometry to 3D to support multi-floor ramp transitions.

- `point_to_segment_projection` — Returns both the distance and the clamped parameter $u$ needed for height interpolation (Eq. 4.5–4.6).
- `ramp_height` — Evaluates $z_{\text{ramp}}(x,y)$ for a given ramp (Eq. 4.7, 4.9).
- `z_surface` — Blends floor and ramp heights per Eq. 5.3–5.4.
- `goal_cost_softmin` — The required soft-min exit selection (Eq. 5.9).

In [2]:
def point_to_segment_projection(x, a, b):
    """
    Extended version of point_to_segment_distance that also returns u,
    the clamped projection parameter needed for ramp height interpolation.
    """
    v = b - a       # eq 4.4: v = B - A, direction vector of ramp centerline
    w = x - a       # eq 4.5: (P - A), vector from ramp start to query point
    vv = np.dot(v, v)   # v · v, denominator of projection formula

    if vv == 0.0:
        return np.linalg.norm(x - a), 0.0

    # eq 4.5: t = (P - A) · v / (v · v)  — raw projection onto infinite line
    t = np.dot(w, v) / vv
    # eq 4.6: u(x,y) = clip(t, 0, 1)  — clamp to finite ramp segment
    t = np.clip(t, 0.0, 1.0)
    q = a + t * v   # closest point on centerline segment
    dist = np.linalg.norm(x - q)   # used for capsule footprint check eq 4.9
    return dist, t  # t is your u for height interpolation in eq 4.7


def ramp_height(x_2d, A, B, z0, z1, r):
    """
    x_2d: (x, y) position of particle
    A, B: ramp centerline endpoints (2D)
    z0: height at A (upper floor)
    z1: height at B (lower floor)
    r: ramp half-width
    Returns: (z_surface, on_ramp)
    """
    dist, u = point_to_segment_projection(x_2d, A, B)
    # eq 4.9: R = {(x,y) | dist((x,y), [A,B]) <= r}  — capsule footprint check
    on_ramp = dist <= r
    # eq 4.7: zramp(x,y) = z0 + (z1 - z0) * u(x,y)  — linear height interpolation
    z_ramp = z0 + (z1 - z0) * u
    return z_ramp, on_ramp


def wall_cost_3d(pos, walls, weight, r):
    """pos is now (x, y, z) but walls are still 2D segments"""
    # walls are defined in the floor plan, so only x,y matter for distance
    x_2d = pos[:2]
    total = 0.0
    # eq 5.13: Cwalls = sum_k w_k * phi(d_k(p))
    for a, b in walls:
        total += wall_penalty_quad(x_2d, a, b, r)
    return total * weight


def z_surface(x_2d, floor_z, ramps):
    """
    Computes the surface height at position x_2d = (x, y).
    ramps: list of dicts with keys A, B, z0, z1, r
    floor_z: the flat floor height for this floor

    Implements eq 5.3: zsurf(x,y) = (1 - g(x,y)) * zfloor + g(x,y) * zramp(x,y)
    where g(x,y) = 1 inside ramp footprint, 0 otherwise  (eq 5.4)
    """
    for ramp in ramps:
        dist, u = point_to_segment_projection(x_2d, ramp['A'], ramp['B'])
        if dist <= ramp['r']:
            # eq 5.4: g(x,y) = 1  — inside ramp footprint, so use ramp height
            # eq 4.7: zramp = z0 + (z1 - z0) * u
            return ramp['z0'] + (ramp['z1'] - ramp['z0']) * u
    # eq 5.4: g(x,y) = 0  — outside all ramps, surface is the flat floor
    return floor_z


def height_cost(pos, floor_z, ramps, wh=50.0):
    """
    Penalizes particle for being at wrong height (eq 5.2)
    pos: (x, y, z)
    """
    x_2d = pos[:2]
    z_target = z_surface(x_2d, floor_z, ramps)
    # eq 5.2: Cheight = wh * (z - zsurf(x,y))^2
    return wh * (pos[2] - z_target)**2


def goal_cost_softmin(pos, exits, tau=1.0):
    """
    exits: list of (x, y, z) exit positions
    tau: temperature parameter (0.5 to 1.5 recommended)

    Replaces hard min with differentiable soft-min (eq 5.9)
    """
    # eq 5.7: C_i(p) = ||p - p_exit_i||^2  — cost to each exit
    costs = [np.sum((pos - e)**2) for e in exits]
    costs = np.array(costs)
    # eq 5.9: Cgoal(p) = -tau * log( sum_i exp(-C_i(p) / tau) )
    # smaller tau -> closer to hard min, larger tau -> blends exits more evenly
    return -tau * np.log(np.sum(np.exp(-costs / tau)))

## 5. Cost Terms — Surface Adherence & Full Cost

This section assembles the remaining cost terms and the full 3D composite cost function.

The **smoothness term** (Eq. 5.14) penalizes abrupt position changes between steps, acting as a temporal regularizer that reduces oscillation.

The **repulsion term** (Eq. 5.15) models crowd interaction using the same quadratic-band $\phi$ as the wall penalty — it activates when two agents come within radius $R$ of each other.

The **numerical gradient** uses central finite differences extended to all three spatial dimensions.

In [3]:
def smoothness_cost(pos_current, pos_previous):
    """
    pos_current:  (x, y, z) position at step k+1
    pos_previous: (x, y, z) position at step k
    """
    # eq 5.14: Csmooth = ||pk+1 - pk||^2
    return np.sum((pos_current - pos_previous)**2)


def repulsion_cost(pos_i, all_positions, i, R=1.0, k_social=5.0):
    """
    pos_i:         (x, y, z) position of particle i
    all_positions: array of all particle positions, shape (N, 3)
    i:             index of current particle (skip self)
    R:             repulsion radius
    k_social:      repulsion strength
    """
    total = 0.0

    for j, pos_j in enumerate(all_positions):
        if j == i:
            continue    # eq 5.15: i != j, skip self

        # phi(||pi - pj||) — quadratic band, same family as wall penalty
        dij = np.linalg.norm(pos_i - pos_j)    # ||pi - pj||
        if dij <= R:
            # phi increases sharply as particles get close — analogous to wall penalty
            total += 0.5 * k_social * (R - dij)**2

    return total


def total_cost_3d(pos, pos_prev, all_positions, particle_idx,
                  exits, walls, floor_z, ramps,
                  tau=1.0, w_wall=5.0, r_wall=0.5,
                  wh=50.0, w_smooth=1.0,
                  r_repulse=1.0, k_social=5.0):
    """
    pos:            (x, y, z) current particle position  — eq 5.1
    pos_prev:       (x, y, z) previous position for smoothness term
    all_positions:  (N, 3) array of all particle positions
    particle_idx:   index of this particle in all_positions
    exits:          list of (x, y, z) ground floor exit positions
    walls:          list of (a, b) 2D wall segments for this floor
    floor_z:        flat floor height the particle should be on
    ramps:          list of ramp dicts {A, B, z0, z1, r}
    tau:            softmin temperature  — eq 5.12
    wh:             height stiffness — tune experimentally per assignment note
    w_smooth:       smoothness weight
    """

    # eq 5.9: Cgoal — soft-min attraction to nearest exit (already implemented)
    c_goal = goal_cost_softmin(pos, exits, tau)

    # eq 5.13: Cwalls — quadratic band wall repulsion (already implemented)
    c_walls = wall_cost_3d(pos, walls, w_wall, r_wall)

    # eq 5.2: Cheight — surface adherence (already implemented)
    c_height = height_cost(pos, floor_z, ramps, wh)

    # eq 5.14: Csmooth — penalizes abrupt jumps between steps
    c_smooth = w_smooth * smoothness_cost(pos, pos_prev)

    # eq 5.15: Crepulsion — short-range crowd repulsion
    c_repulsion = repulsion_cost(pos, all_positions, particle_idx, r_repulse, k_social)

    # eq 5.6: C = Cgoal + Cwalls + Cheight + Csmoothness + Crepulsion
    return c_goal + c_walls + c_height + c_smooth + c_repulsion


def numerical_gradient_3d(pos, pos_prev, all_positions, particle_idx,
                           exits, walls, floor_z, ramps,
                           epsilon=1e-4, **cost_params):
    """
    Central finite differences over all 3 dimensions (x, y, z)
    grad[dim] = (C(pos + eps) - C(pos - eps)) / (2 * eps)
    """
    # 3D now — was range(2) in your original gradient_social_numerical
    grad = np.zeros(3)

    for dim in range(3):
        pos_plus = pos.copy()
        pos_minus = pos.copy()
        pos_plus[dim] += epsilon
        pos_minus[dim] -= epsilon

        c_plus = total_cost_3d(pos_plus, pos_prev, all_positions,
                               particle_idx, exits, walls, floor_z, ramps,
                               **cost_params)
        c_minus = total_cost_3d(pos_minus, pos_prev, all_positions,
                                particle_idx, exits, walls, floor_z, ramps,
                                **cost_params)

        # central finite difference approximation
        grad[dim] = (c_plus - c_minus) / (2 * epsilon)

    return grad




    

## 6. Robot Arm — IK Framework

This section introduces the robotic agent into the simulation. The robot is a 4-DOF arm whose end-effector moves through the building as a dynamic obstacle.

The arm is defined by four joint angles $\boldsymbol{\Phi} = [\phi_1, \phi_2, \phi_3, \phi_4]$ and four link lengths $L_1 \ldots L_4$. The end-effector world position $\mathbf{e}$ is produced by composing homogeneous transforms:

$$T_{04} = T_{01}\, T_{12}\, T_{23}\, T_{34}$$

$$\mathbf{e} = T_{04}[{:}3,\; 3]$$

The **numerical Jacobian** maps small joint-angle changes to end-effector displacement:

$$J_{[:,k]} = \frac{\mathbf{e}(\boldsymbol{\Phi} + \delta_k) - \mathbf{e}(\boldsymbol{\Phi})}{\delta\phi}$$

where $\delta_k$ perturbs only joint $k$ by $\delta\phi = 0.1°$.

**IK update rule** (Jacobian pseudo-inverse):

$$\Delta\boldsymbol{\phi} = J^{\dagger}\, \lambda\,(\mathbf{p}_{\text{target}} - \mathbf{e}), \qquad J^{\dagger} = J^{\top}(JJ^{\top})^{-1}$$

The IK is run for up to 60 iterations per simulation step, giving the arm a limited motion budget that keeps runtime comparable to Assignment 1.

**End-effector as obstacle:** Once $\mathbf{e}(t)$ is known, a quadratic-band repulsion cost is added to every particle's cost function:

$$C_{\text{robot}}(\mathbf{p}_i) = k_{\text{ee}} \cdot \max\!\left(0,\; r_{\text{ee}} - \|\mathbf{p}_i - \mathbf{e}(t)\|\right)^2 \tag{10.1}$$

This mirrors the wall penalty (Eq. 5.13) so the existing gradient machinery handles the robot obstacle exactly like a wall segment — no special-case code required.

- `RobotArm` — class encapsulating forward kinematics, Jacobian, IK solver, and vedo rendering.
- `robot_obstacle_cost` — Eq. 10.1, the quadratic-band repulsion from the end-effector.

In [4]:
import vedo
from vedo import Sphere, Plotter, LinearTransform

vedo.settings.default_backend = 'vtk'


# =============================================================
# ROBOT OBSTACLE COST — quadratic-band repulsion (eq 10.1)
# Same phi(d) family as wall_penalty_quad, applied to the
# end-effector sphere instead of a wall segment.
# =============================================================

def robot_obstacle_cost(pos, ee_pos, r_ee=1.5, k_ee=8.0):
    """
    pos:    (x, y, z) particle position
    ee_pos: (x, y, z) end-effector world position
    r_ee:   repulsion radius — particles inside this sphere are pushed away
    k_ee:   repulsion strength

    eq 10.1: C_robot = k_ee * 0.5 * (r_ee - d)^2  if d <= r_ee, else 0
    """
    d = np.linalg.norm(pos - ee_pos)
    # eq 10.1: same quadratic-band as wall penalty, centred on end-effector
    if d <= r_ee:
        return 0.5 * k_ee * (r_ee - d)**2
    return 0.0


# =============================================================
# ROBOT ARM CLASS
# 4-DOF arm: base rotation (z) + 3 pitch joints (y).
# Lengths are scaled down from the starter notebook (mm) to
# match the 12×10 building floor plan (world units).
# =============================================================

class RobotArm:
    """
    Robot arm with:
      - forward kinematics (homogeneous transforms)
      - numerical Jacobian
      - Jacobian pseudo-inverse IK for continuous target tracking
      - vedo cylinder rendering
    """

    def __init__(self, partLengths, arm_location):
        self.arm_location = np.array(arm_location, dtype=float).ravel()

        # Link lengths
        self.L1, self.L2, self.L3, self.L4 = partLengths

        # IK tuning — kept small so each timestep is fast
        self.delta_phi        = 0.1     # finite-difference step (degrees)
        self.target_lambda    = 0.25    # IK step size — increased for faster convergence
        self.target_tolerance = 0.3     # stop when end-effector is within this distance
        self.convergence      = 0.01    # stop when joint change is smaller than this
        self.iteration_limit  = 8     # max IK iterations per simulation step

        # Current joint state (degrees)
        self.phi = np.zeros(4, dtype=float)

    # ── rotation matrix helpers ─────────────────────────────

    @staticmethod
    def _rot(theta, axis):
        """Single-axis rotation matrix for angle theta (degrees)."""
        c = np.cos(np.radians(theta))
        s = np.sin(np.radians(theta))
        if axis == 'x':
            return np.array([[1,0,0],[0,c,-s],[0,s,c]])
        if axis == 'y':
            return np.array([[c,0,s],[0,1,0],[-s,0,c]])
        if axis == 'z':
            return np.array([[c,-s,0],[s,c,0],[0,0,1]])
        raise ValueError(f'Unknown axis: {axis}')

    @staticmethod
    def _T(R, t):
        """Homogeneous transform from rotation R and translation t."""
        M = np.eye(4)
        M[:3, :3] = R
        M[:3,  3] = np.array(t).ravel()
        return M

    # ── forward kinematics ──────────────────────────────────

    def forward_kinematics(self, Phi):
        """
        Phi: (4,) joint angles in degrees.
        Returns T00, T01, T02, T03, T04, e
        where e = T04[:3, 3] is the end-effector world position.
        """
        r = 0.4  # joint sphere radius (cosmetic offset)

        # Base frame — no rotation, just position
        T00 = self._T(self._rot(0, 'z'),
                      [self.arm_location[0], self.arm_location[1], 0])

        # Joint 1 — rotation about z (yaw)
        T01 = self._T(self._rot(Phi[0], 'z'), self.arm_location)

        # Joint 2 — rotation about y (pitch), offset by L1 along z
        T12 = self._T(self._rot(Phi[1], 'y'), [0, 0, self.L1 + 2*r])
        T02 = T01 @ T12

        # Joint 3 — rotation about y (pitch), offset by L2 along z
        T23 = self._T(self._rot(Phi[2], 'y'), [0, 0, self.L2 + 2*r])
        T03 = T02 @ T23

        # End-effector frame — offset by L3 along z (and small x offset)
        T34 = self._T(self._rot(Phi[3], 'y'), [-28.4 * (self.L3/416), 0, self.L3 + r])
        T04 = T03 @ T34

        e = T04[:3, 3]
        return T00, T01, T02, T03, T04, e

    def end_effector_pos(self):
        """Returns current end-effector world position."""
        return self.forward_kinematics(self.phi)[-1]

    # ── Jacobian ────────────────────────────────────────────

    def jacobian(self, phi):
        """
        Numerical Jacobian J (3×3) mapping first 3 joint velocities
        to end-effector velocity.  Uses forward finite differences.
        """
        e = self.forward_kinematics(phi)[-1]
        cols = []
        for k in range(3):   # only 3 joints affect position; joint 4 is a wrist roll
            dp = np.zeros(4); dp[k] = self.delta_phi
            e2 = self.forward_kinematics(phi + dp)[-1]
            cols.append((e2 - e) / self.delta_phi)
        return np.column_stack(cols)   # (3, 3)

    # ── IK solver ───────────────────────────────────────────

    def ik_step(self, target_world):
        """
        Run up to iteration_limit IK iterations driving the end-effector
        toward target_world (3D world-space point).
        Updates self.phi in-place and returns the new joint angles.
        """
        target = np.array(target_world, dtype=float)
        for _ in range(self.iteration_limit):
            _, _, _, _, _, e = self.forward_kinematics(self.phi)
            err = target - e
            if np.linalg.norm(err) < self.target_tolerance:
                break
            J      = self.jacobian(self.phi)
            J_pinv = np.linalg.pinv(J)           # pseudo-inverse
            dphi3  = J_pinv @ (self.target_lambda * err)
            delta  = np.append(dphi3, 0.0)       # joint 4 unchanged
            self.phi = self.phi + delta
        return self.phi.copy()

    # ── rendering ───────────────────────────────────────────

    def render_meshes(self, ee_radius=1.5):
        """Return a list of vedo objects representing the arm at current phi."""
        T00, T01, T02, T03, T04, e = self.forward_kinematics(self.phi)
        objects = []

        origin  = T01[:3,3]
        joint2  = T02[:3,3]
        joint3  = T03[:3,3]
        ee      = T04[:3,3]

        def seg(p1, p2, col, r=1.5):
            if np.linalg.norm(p2-p1) < 0.01:
                return None
            return vedo.Cylinder(pos=[p1, p2], r=r, res=12, c=col, alpha=0.9)

        s0 = seg(self.arm_location.ravel(), origin, 'blue5', r=2.0)
        s1 = seg(origin, joint2, 'lightblue', r=1.8)
        s2 = seg(joint2, joint3, 'gray5', r=1.5)
        s3 = seg(joint3, ee, 'red5', r=1.2)
        ee_sphere = Sphere(pos=ee, r=ee_radius * 0.35, c='orange', alpha=0.85)

        for obj in [s0, s1, s2, s3, ee_sphere]:
            if obj is not None:
                objects.append(obj)
        return objects, ee

## 7. Clustering & Prediction

To decide where to move, the robot must identify where evacuation is happening. This section implements two components: a k-means clusterer that groups ground-floor particles, and a centroid velocity model that predicts where the dominant cluster will be in the near future.

**Why ground-floor only?** Particles on upper floors are still descending ramps and have not yet committed to an exit. Filtering to `z < 0.5·H` means the robot only reacts to agents that are actively approaching an exit — the same logic used to declare an agent evacuated.

### 7.1 k-means (Lloyd's algorithm, k = 2)

$$\min_{\{C_k\}} \sum_{k=1}^{K} \sum_{\mathbf{p}_i \in C_k} \|\mathbf{p}_i^{(xy)} - \boldsymbol{\mu}_k\|^2$$

k = 2 is used because there are two exits; in most frames the two clusters map naturally onto the two exit streams. Clustering is done in 2D (x, y only) since all ground-floor agents share roughly the same z.

The **dominant cluster** is whichever cluster has the most members — this represents the largest active escape stream and is the robot's primary target.

### 7.2 Cluster velocity & linear prediction

Centroid velocity is estimated by finite difference between the current and previous centroid:

$$\mathbf{v}_t = \boldsymbol{\mu}_t - \boldsymbol{\mu}_{t-1}$$

The predicted interception point at horizon $T$ steps ahead:

$$\hat{\mathbf{p}}(t + T) = \boldsymbol{\mu}_t + T \cdot \mathbf{v}_t \tag{7.1}$$

This is a simple constant-velocity extrapolation. It is sufficient because particles near exits move fairly steadily toward the exit and their centroid velocity changes slowly.

- `kmeans_2d` — Lloyd's k-means in 2D, no external dependencies.
- `ClusterTracker` — stateful wrapper that maintains centroid history and exposes the prediction (Eq. 7.1).

In [5]:
# =============================================================
# SECTION 7: CLUSTERING & PREDICTION
# =============================================================

def kmeans_2d(points, k=2, n_iter=20, rng=None):
    """
    Lloyd's k-means clustering in 2D.

    points: (N, 2) array of particle xy positions
    k:      number of clusters
    Returns: labels (N,), centroids (k, 2)
    """
    if rng is None:
        rng = np.random.default_rng(0)
    N = len(points)

    # edge cases: not enough points to form k clusters
    if N == 0:
        return np.array([]), np.zeros((k, 2))
    if N <= k:
        labels    = np.arange(N)
        centroids = np.zeros((k, 2))
        centroids[:N] = points.copy()
        return labels, centroids

    # initialise centroids with k randomly chosen distinct points
    idx       = rng.choice(N, k, replace=False)
    centroids = points[idx].copy().astype(float)

    for _ in range(n_iter):
        # assignment step: each point takes the nearest centroid
        # dists shape: (N, k)
        dists  = np.linalg.norm(points[:, None, :] - centroids[None, :, :], axis=2)
        labels = np.argmin(dists, axis=1)

        # update step: recompute centroid as mean of assigned points
        new_c = centroids.copy()
        for c in range(k):
            members = points[labels == c]
            if len(members) > 0:
                new_c[c] = members.mean(axis=0)

        # stop early if centroids have not moved
        if np.allclose(centroids, new_c, atol=1e-6):
            break
        centroids = new_c

    return labels, centroids


def select_dominant_cluster(labels, k):
    """Return index of the largest cluster (most assigned particles)."""
    counts = np.array([np.sum(labels == c) for c in range(k)])
    return int(np.argmax(counts))


class ClusterTracker:
    """
    Stateful tracker for the dominant ground-floor cluster.

    Maintains centroid history across simulation steps and exposes
    a linear velocity prediction (eq 11.1).
    """

    def __init__(self, k=2, pred_horizon=8):
        self.k            = k
        self.pred_horizon = pred_horizon   # T in eq 11.1 — steps to predict ahead
        self._prev_mu     = None           # centroid at t-1
        self._mu          = None           # centroid at t
        self.velocity     = np.zeros(2)   # estimated centroid velocity
        self._rng         = np.random.default_rng(1)

    def update(self, positions, evacuated, H):
        """
        Run clustering on ground-floor particles and update velocity estimate.

        positions:  (N, 3) all particle positions
        evacuated:  (N,)  bool mask
        H:          floor height constant
        Returns:    centroid (2,), predicted (2,), labels, centroids, dom_idx
        """
        # filter to ground floor only: z < 0.5 * H and not yet evacuated
        mask = (~evacuated) & (positions[:, 2] < H * 0.5)
        pts  = positions[mask, :2]   # (M, 2) — xy positions only

        # not enough particles to cluster: return last known position
        if len(pts) < 2:
            fallback = self._mu if self._mu is not None else np.array([6.0, 2.0])
            return fallback, fallback, np.array([]), np.zeros((self.k, 2)), 0

        labels, centroids = kmeans_2d(pts, k=self.k, rng=self._rng)
        dom_idx  = select_dominant_cluster(labels, self.k)
        centroid = centroids[dom_idx].copy()

        # estimate centroid velocity by finite difference (eq 11.1 numerator)
        if self._mu is not None:
            self.velocity = centroid - self._mu
        self._prev_mu = self._mu
        self._mu      = centroid

        # eq 11.1: predicted position = mu + T * v
        predicted = centroid + self.pred_horizon * self.velocity
        return centroid, predicted, labels, centroids, dom_idx

## 8. Robot-Interference Simulation

This is the main combined simulation cell. It reuses all parameters, geometry, and cost functions from the cost functions defined in Sections 3–5 without modification. The only additions are:

1. `robot_obstacle_cost` (Eq. 10.1) is added inside `_cost_for_agent` so every particle repels from the end-effector.
2. A `RobotArm` instance is mounted outside the building at $(6, -2, 0)$, centred in x between the two exits.
3. A `ClusterTracker` runs every 5 steps to identify the dominant evacuation stream.
4. The robot calls `ik_step` each simulation step to move its end-effector toward the predicted cluster position.

**Rendering additions:**
- **Cyan sphere** — current cluster centroid.
- **Yellow sphere** — predicted intercept position (Eq. 7.1).
- **Orange sphere** — robot end-effector (radius = `R_EE`, matching the repulsion zone).
- Robot arm drawn as grey cylinders.

| New Parameter | Value | Role |
|---|---|---|
| `R_EE` | 1.5 | End-effector repulsion radius |
| `K_EE` | 8.0 | End-effector repulsion strength |
| `PRED_HORIZON` | 8 | Prediction horizon T (steps, Eq. 7.1) |
| `CLUSTER_K` | 2 | Number of k-means clusters |
| `CLUSTER_EVERY` | 5 | Re-cluster every N steps |

In [6]:
# =============================================================
# SECTION 8: ROBOT-INTERFERENCE SIMULATION
# All cost parameters from Sections 3–5 are reused unchanged.
# New parameters for the robot are defined at the top.
# =============================================================

import numpy as np
import os, glob
import vedo
vedo.settings.default_backend = 'vtk'

# ── simulation parameters ──────────────────────────────────
N_AGENTS    = 25
N_STEPS     = 400
FRAME_EVERY = 3

STEP_SIZE = 0.08
MAX_STEP  = 0.20
MOMENTUM  = 0.30
EPSILON   = 1e-4
EXIT_TOL  = 1.2

TAU       = 1.0
W_GOAL    = 5.0
W_WALL    = 10.0
R_WALL    = 0.65
WH        = 60.0
W_SMOOTH  = 0.15
R_REPULSE = 1.2
K_SOCIAL  = 3.5
W_GRAVITY = 45.0

H       = 5.0
FLOOR_W = 12.0
FLOOR_D = 10.0
FLOOR_T = 0.15
WALL_T  = 0.25
WALL_H  = 2.0
RAMP_W  = 0.5
Z       = [0.0, H, 2.0 * H]

EXITS_3D = [np.array([2.0, 0.3, Z[0]]), np.array([10.0, 0.3, Z[0]])]
EXITS_2D = [np.array([2.0, 0.3]),        np.array([10.0, 0.3])]

RAMPS = [
    {'A': np.array([2.0,  3.0]), 'B': np.array([2.0,  8.0]),  'z0': Z[1], 'z1': Z[0], 'r': RAMP_W},
    {'A': np.array([10.0, 3.0]), 'B': np.array([10.0, 8.0]),  'z0': Z[1], 'z1': Z[0], 'r': RAMP_W},
    {'A': np.array([2.0,  3.0]), 'B': np.array([6.0,  3.0]),  'z0': Z[2], 'z1': Z[1], 'r': RAMP_W},
    {'A': np.array([2.0,  8.0]), 'B': np.array([10.0, 8.0]),  'z0': Z[2], 'z1': Z[1], 'r': RAMP_W},
]

FLOOR_TARGETS = {
    2: [np.array([2., 3.]), np.array([2., 8.])],
    1: [np.array([2., 5.5]), np.array([10., 5.5])],
    0: EXITS_2D,
}

_W = FLOOR_W; _D = FLOOR_D
WALLS_PER_FLOOR = {
    0: [(np.array([0.,0.]),np.array([1.2,0.])),(np.array([2.8,0.]),np.array([9.2,0.])),(np.array([10.8,0.]),np.array([_W,0.])),(np.array([0.,_D]),np.array([_W,_D])),(np.array([0.,0.]),np.array([0.,_D])),(np.array([_W,0.]),np.array([_W,_D]))],
    1: [(np.array([0.,0.]),np.array([1.2,0.])),(np.array([2.8,0.]),np.array([9.2,0.])),(np.array([10.8,0.]),np.array([_W,0.])),(np.array([0.,_D]),np.array([_W,_D])),(np.array([0.,0.]),np.array([0.,_D])),(np.array([_W,0.]),np.array([_W,_D])),(np.array([6.,0.]),np.array([6.,4.])),(np.array([6.,6.]),np.array([6.,_D]))],
    2: [(np.array([0.,0.]),np.array([_W,0.])),(np.array([0.,_D]),np.array([_W,_D])),(np.array([0.,0.]),np.array([0.,_D])),(np.array([_W,0.]),np.array([_W,_D])),(np.array([0.,5.]),np.array([4.,5.])),(np.array([6.,5.]),np.array([_W,5.]))],
}

# ── : robot / end-effector parameters ─────────────────────
R_EE          = 2   # end-effector repulsion radius (eq 10.1)
K_EE          = 100.0    # end-effector repulsion strength (eq 10.1)
PRED_HORIZON  = 6      # prediction horizon T in steps (eq 11.1)
CLUSTER_K     = 2     # number of k-means clusters
CLUSTER_EVERY = 5      # re-run clustering every N steps

# ── particle cost (Sections 3–5 cost + robot obstacle) ──────────

def z_surface_global(x_2d, z_current):
    # unchanged from Sections 3–5
    ramp_candidates = []
    for ramp in RAMPS:
        dist, u = point_to_segment_projection(x_2d, ramp['A'], ramp['B'])
        if dist > ramp['r']:
            continue
        z_ramp = ramp['z0'] + (ramp['z1'] - ramp['z0']) * u
        z_hi   = max(ramp['z0'], ramp['z1'])
        if abs(z_current - z_hi) <= H * 0.7:
            ramp_candidates.append(z_ramp)
    if ramp_candidates:
        return min(ramp_candidates, key=lambda z: abs(z - z_current))
    return min(Z, key=lambda z: abs(z - z_current))

def get_targets(pos):
    # unchanged from Sections 3–5
    x_2d = pos[:2]; z = pos[2]
    best = None; best_d = RAMP_W + 1.0
    for ramp in RAMPS:
        dist, u = point_to_segment_projection(x_2d, ramp['A'], ramp['B'])
        if dist > ramp['r']:
            continue
        z_hi = max(ramp['z0'], ramp['z1'])
        if abs(z - z_hi) <= H * 0.7 and dist < best_d:
            best_d = dist; best = ramp
    if best is not None:
        return [best['B']]
    if z < H * 0.5:   return FLOOR_TARGETS[0]
    elif z < H * 1.5: return FLOOR_TARGETS[1]
    else:              return FLOOR_TARGETS[2]

def z_to_floor(z):
    if z < H * 0.5:   return 0
    elif z < H * 1.5: return 1
    else:              return 2

def _height_cost(pos):
    return WH * (pos[2] - z_surface_global(pos[:2], pos[2])) ** 2

def _gravity_cost(pos):
    return W_GRAVITY * pos[2]

def _goal_cost(pos):
    targets = get_targets(pos)
    costs   = np.array([np.sum((pos[:2] - t) ** 2) for t in targets])
    cs      = costs - costs.min()
    return W_GOAL * (-TAU * np.log(np.sum(np.exp(-cs / TAU))) + costs.min() / TAU)

def _cost_for_agent(pos, pos_prev, all_pos, idx, ee_pos):
    """
    Full per-agent cost including the new robot obstacle term (eq 10.1).
    ee_pos: current end-effector world position — None disables the robot term.
    """
    fi = z_to_floor(pos[2])
    c = (_goal_cost(pos)
       + wall_cost_3d(pos, WALLS_PER_FLOOR[fi], W_WALL, R_WALL)
       + _height_cost(pos)
       + _gravity_cost(pos)
       + W_SMOOTH * smoothness_cost(pos, pos_prev)
       + repulsion_cost(pos, all_pos, idx, R_REPULSE, K_SOCIAL)
       + robot_obstacle_cost(pos, ee_pos, R_EE, K_EE))   # eq 10.1 — NEW
    return c

def _gradient(pos, pos_prev, all_pos, idx, ee_pos):
    # central finite differences — identical structure to Section 5
    grad = np.zeros(3)
    for dim in range(3):
        pp = pos.copy(); pp[dim] += EPSILON
        pm = pos.copy(); pm[dim] -= EPSILON
        grad[dim] = (_cost_for_agent(pp, pos_prev, all_pos, idx, ee_pos) -
                     _cost_for_agent(pm, pos_prev, all_pos, idx, ee_pos)) / (2 * EPSILON)
    return grad

# ── initial particle positions ──────────────────────────────
rng = np.random.default_rng(42)
def _rp(fi):
    m = 1.5
    return np.array([rng.uniform(m, FLOOR_W-m), rng.uniform(m, FLOOR_D-m), Z[fi]], dtype=float)

positions      = np.array([_rp(2) for _ in range(9)] + [_rp(1) for _ in range(8)] + [_rp(0) for _ in range(8)], dtype=float)
prev_positions = positions.copy()
velocities     = np.zeros_like(positions)
evacuated      = np.zeros(N_AGENTS, dtype=bool)
AGENT_COLORS   = ['#e74c3c']*9 + ['#3498db']*8 + ['#2ecc71']*8
AGENT_R        = 0.28

# ── robot arm setup ───────────────────────────────────────────
# Scale the starter link lengths (mm) down to match the 12×10 building.
# ARM_SCALE chosen so L3 (longest link) ≈ 7.5 world units,
# giving the arm enough reach to cover the ground floor exit area.
ARM_SCALE = 1.0 / 55.0
L = [81*ARM_SCALE, 217*ARM_SCALE, 416*ARM_SCALE, 0.0]

# Base mounted at (6, -2, 0) — centred in x, just in front of the front wall.
# This gives a clear line-of-sight to both exits without entering the building.
arm_base = np.array([6.0, -2.0, 0.0], dtype=float)
robot    = RobotArm(L, arm_base)
robot.phi = np.array([0.0, -30.0, 20.0, 0.0])   # reasonable starting pose

# ── cluster tracker ───────────────────────────────────────────
tracker = ClusterTracker(k=CLUSTER_K, pred_horizon=PRED_HORIZON)

# ── static building objects ─────────────────────────────────
def _build_static_objects():
    objs = []
    for fi, col in enumerate(['#d0d0d0','#c8ddf0','#f5f0d0']):
        b=vedo.Box(pos=(FLOOR_W/2,FLOOR_D/2,Z[fi]-FLOOR_T/2),length=FLOOR_W,width=FLOOR_D,height=FLOOR_T)
        b.color(col).alpha(0.6); objs.append(b)
    def _hw(x0,x1,y,fi,h=WALL_H,al=1.0):
        bx=vedo.Box(pos=((x0+x1)/2,y,Z[fi]+h/2),length=abs(x1-x0),width=WALL_T,height=h)
        bx.color('white').alpha(al); return bx
    def _vw(x,y0,y1,fi,h=WALL_H,al=1.0):
        bx=vedo.Box(pos=(x,(y0+y1)/2,Z[fi]+h/2),length=WALL_T,width=abs(y1-y0),height=h)
        bx.color('white').alpha(al); return bx
    for fi in range(3):
        objs += [_hw(0,FLOOR_W,FLOOR_D,fi,h=2.8,al=1.0),_vw(0,0,FLOOR_D,fi,al=1.0),
                 _vw(FLOOR_W,0,FLOOR_D,fi,al=0.10),_hw(0,1.2,0,fi,al=0.10),
                 _hw(2.8,9.2,0,fi,al=0.10),_hw(10.8,FLOOR_W,0,fi,al=0.10)]
    objs += [_vw(6,0,4,1,al=0.75),_vw(6,6,FLOOR_D,1,al=0.75),
             _hw(0,4,5,2,al=0.75),_hw(6,FLOOR_W,5,2,al=0.75)]
    for ramp in RAMPS:
        A,B,z0,z1,r=ramp['A'],ramp['B'],ramp['z0'],ramp['z1'],ramp['r']
        v=B-A; vn=v/np.linalg.norm(v); perp=np.array([-vn[1],vn[0]])*r
        n=40; verts=[]; faces=[]
        for i in range(n+1):
            tt=i/n; zz=z0+(z1-z0)*tt; mid=A+tt*v
            verts+=[[mid[0]-perp[0],mid[1]-perp[1],zz],[mid[0]+perp[0],mid[1]+perp[1],zz]]
        for i in range(n):
            il=2*i; faces.append([il,il+1,il+3,il+2])
        m=vedo.Mesh([verts,faces]); m.color('sandybrown').alpha(1.0).lw(1).lc('saddlebrown'); objs.append(m)
    for xy in [[2.,0.3],[10.,0.3]]:
        s=vedo.Sphere(pos=(xy[0],xy[1],Z[0]+0.3),r=0.35); s.color('lime'); objs.append(s)
        d=vedo.Disc(pos=(xy[0],xy[1],Z[0]+0.02),r1=0,r2=0.7,res=24); d.color('lime').alpha(0.7); objs.append(d)
    return objs

static_objects = _build_static_objects()

FRAME_DIR = '/tmp/evac_robot_frames'
os.makedirs(FRAME_DIR, exist_ok=True)
for f in glob.glob(f'{FRAME_DIR}/*.png'): os.remove(f)
frame_idx = 0

# ── render helper ─────────────────────────────────────────────
def render_frame(step, centroid, predicted):
    global frame_idx
    plt = vedo.Plotter(offscreen=True, size=(1200, 800), bg='white')
    plt.add(static_objects)

    # particles
    for i, (pos, evac) in enumerate(zip(positions, evacuated)):
        if evac: continue
        s = vedo.Sphere(pos=(pos[0], pos[1], pos[2]+AGENT_R), r=AGENT_R)
        s.color(AGENT_COLORS[i]).alpha(1.0); plt.add(s)

    # cluster centroid (cyan) and prediction (yellow)
    if centroid is not None:
        plt.add(vedo.Sphere(pos=(centroid[0], centroid[1], Z[0]+0.5), r=0.5, c='cyan', alpha=0.8))
        plt.add(vedo.Sphere(pos=(predicted[0], predicted[1], Z[0]+0.5), r=0.4, c='yellow', alpha=0.8))

    # robot arm + end-effector sphere
    arm_meshes, _ = robot.render_meshes(ee_radius=R_EE)
    for m in arm_meshes: plt.add(m)

    n_evac = int(evacuated.sum())
    plt.add(vedo.Text2D(
        f"Step {step:>3d}/{N_STEPS}   Evacuated: {n_evac}/{N_AGENTS}   "
        f"[Red=F2  Blue=F1  Green=F0  Cyan=Cluster  Yellow=Prediction  Orange=EE]",
        pos='top-left', s=0.65, c='black'))

    plt.camera.SetPosition(FLOOR_W*2.5, -FLOOR_D*1.5, H*5.5)
    plt.camera.SetFocalPoint(FLOOR_W/2, FLOOR_D/2, H)
    plt.camera.SetViewUp(0, 0, 1)
    plt.render()
    plt.screenshot(f'{FRAME_DIR}/frame_{frame_idx:04d}.png')
    plt.close()
    frame_idx += 1

# ── main simulation loop ──────────────────────────────────────
print("Starting robot-interference simulation...")
print(f"  {N_AGENTS} agents | {N_STEPS} steps | frame every {FRAME_EVERY} steps")

centroid  = None
predicted = np.array([6.0, 1.0])   # fallback before first cluster update
render_frame(0, centroid, predicted)

for step in range(1, N_STEPS + 1):

    # 1. Cluster update (every CLUSTER_EVERY steps)
    if step % CLUSTER_EVERY == 0:
        result   = tracker.update(positions, evacuated, H)
        centroid, predicted = result[0], result[1]

    # 2. Move robot toward predicted cluster position
    # Clamp prediction to building bounds so the arm never chases a point outside
    target_xy = np.clip(predicted, [0.3, 0.3], [FLOOR_W-0.3, FLOOR_D-0.3])
    target_3d = np.array([target_xy[0], target_xy[1], Z[0] + 0.6])
    
    if not hasattr(robot, '_smooth_target'):
        robot._smooth_target = target_3d.copy()
    robot._smooth_target = 0.06 * target_3d + 0.94 * robot._smooth_target
    robot.ik_step(robot._smooth_target)

    
    ee_pos = robot.end_effector_pos()
    

    # 3. Particle update — same structure as Sections 3–5 except ee_pos is passed in
    all_pos = positions.copy()
    for i in range(N_AGENTS):
        if evacuated[i]: continue
        pos  = positions[i]
        prev = prev_positions[i]

        grad = _gradient(pos, prev, all_pos, i, ee_pos)
        velocities[i] = MOMENTUM * velocities[i] - STEP_SIZE * grad
        spd = np.linalg.norm(velocities[i])
        if spd > MAX_STEP:
            velocities[i] = velocities[i] / spd * MAX_STEP

        prev_positions[i] = positions[i].copy()
        positions[i]      = positions[i] + velocities[i]

        # xy boundary clamp
        positions[i][0] = np.clip(positions[i][0], 0.1, FLOOR_W - 0.1)
        positions[i][1] = np.clip(positions[i][1], 0.1, FLOOR_D - 0.1)

        # hard z-clamp — prevents floor phasing
        z_surf = z_surface_global(positions[i][:2], positions[i][2])
        if positions[i][2] < z_surf:
            positions[i][2]  = z_surf
            velocities[i][2] = max(velocities[i][2], 0.0)

        positions[i][2] = np.clip(positions[i][2], -0.1, 2*H + 0.1)

        # evacuation check
        for ex in EXITS_3D:
            if np.linalg.norm(positions[i][:2] - ex[:2]) < EXIT_TOL and positions[i][2] < H*0.3:
                evacuated[i] = True; break

    if step % FRAME_EVERY == 0 or step == N_STEPS:
        ee = robot.end_effector_pos()
        print(f"  Step {step:>3d}: {int(evacuated.sum())}/{N_AGENTS} evacuated  |  EE @ ({ee[0]:.1f}, {ee[1]:.1f}, {ee[2]:.1f})")
        render_frame(step, centroid, predicted)

print(f"\nDone. {int(evacuated.sum())}/{N_AGENTS} agents evacuated.")
print(f"{frame_idx} frames saved to {FRAME_DIR}")
print("Run the next cell to encode the video.")

Starting robot-interference simulation...
  25 agents | 400 steps | frame every 3 steps
  Step   3: 0/25 evacuated  |  EE @ (6.1, 1.0, 0.4)
  Step   6: 0/25 evacuated  |  EE @ (6.1, 1.1, 0.5)
  Step   9: 0/25 evacuated  |  EE @ (6.2, 1.4, 0.5)
  Step  12: 0/25 evacuated  |  EE @ (5.5, 2.2, 0.6)
  Step  15: 0/25 evacuated  |  EE @ (4.7, 2.7, 0.6)
  Step  18: 0/25 evacuated  |  EE @ (4.1, 2.5, 0.6)
  Step  21: 0/25 evacuated  |  EE @ (3.6, 2.2, 0.6)
  Step  24: 1/25 evacuated  |  EE @ (3.2, 1.8, 0.6)
  Step  27: 1/25 evacuated  |  EE @ (2.7, 1.6, 0.6)
  Step  30: 1/25 evacuated  |  EE @ (2.4, 1.5, 0.6)
  Step  33: 2/25 evacuated  |  EE @ (2.1, 1.9, 0.6)
  Step  36: 2/25 evacuated  |  EE @ (2.0, 2.1, 0.6)
  Step  39: 3/25 evacuated  |  EE @ (1.8, 2.3, 0.6)
  Step  42: 3/25 evacuated  |  EE @ (1.6, 2.9, 0.6)
  Step  45: 3/25 evacuated  |  EE @ (1.4, 3.3, 0.6)
  Step  48: 3/25 evacuated  |  EE @ (1.4, 3.2, 0.6)
  Step  51: 4/25 evacuated  |  EE @ (2.4, 3.5, 0.6)
  Step  54: 5/25 evacuated  

## 9. Robot Video Export

This cell encodes the frames saved in `/tmp/evac_robot_frames/` into a single MP4 file.
Run it after Section 8 finishes.

In [7]:
# =============================================================
# SECTION 9: ROBOT VIDEO EXPORT — frames → MP4
# Run this cell after Section 8 finishes.
# =============================================================

import imageio.v2 as iio
import glob, os
from IPython.display import Video, display

FRAME_DIR  = '/tmp/evac_robot_frames'
OUTPUT_MP4 = 'robot_evacuation.mp4'
FPS        = 12

frame_files = sorted(glob.glob(f'{FRAME_DIR}/frame_*.png'))
print(f"Found {len(frame_files)} frames → encoding at {FPS} fps...")

writer = iio.get_writer(OUTPUT_MP4, fps=FPS, quality=8)
for fp in frame_files:
    writer.append_data(iio.imread(fp))
writer.close()

size_mb = os.path.getsize(OUTPUT_MP4) / 1e6
print(f"Saved: {OUTPUT_MP4}  ({size_mb:.1f} MB)")
display(Video(OUTPUT_MP4, embed=True, width=900))

Found 135 frames → encoding at 12 fps...
Saved: robot_evacuation.mp4  (0.8 MB)


## 10. Robot Discussion & Results

### 10.1 Clustering Methods

To determine where to move, the robot uses **k-means clustering (k = 2)** on the positions of ground-floor particles — those with $z < 0.5H$. Restricting clustering to the ground floor ensures the robot only reacts to agents who are actively approaching an exit, ignoring particles still descending from upper floors.

**Lloyd's algorithm** is used with 20 update iterations per clustering call. Each particle is assigned to the nearest of two centroids; centroids are then recomputed as the mean of their assigned members. The algorithm stops early if centroids converge to within $10^{-6}$ tolerance. $k = 2$ is used because there are two exits — in most frames the two clusters map naturally onto the two distinct exit streams.

The **dominant cluster** is the one with the largest membership. This represents the most active evacuation stream and is the robot's primary target each step. Re-clustering runs every 5 simulation steps (`CLUSTER_EVERY = 5`) to keep runtime manageable.

### 10.2 Velocity Prediction

Rather than chasing the current cluster centroid, the robot predicts where the dominant cluster will be in the near future using a **constant-velocity linear model** (Eq. 7.1):

$$\hat{\mathbf{p}}(t + T) = \boldsymbol{\mu}_t + T \cdot \mathbf{v}_t, \qquad \mathbf{v}_t = \boldsymbol{\mu}_t - \boldsymbol{\mu}_{t-1}$$

A prediction horizon of $T = 6$ steps (`PRED_HORIZON = 6`) is used. This simple extrapolation is sufficient because ground-floor particles move steadily and predictably toward the exits once they reach the ground floor.

### 10.3 Robot Behavior

The 4-DOF arm is mounted at $(6, -2, 0)$, centred between the two exits just outside the front wall. At each simulation step the IK solver drives the end-effector toward the predicted intercept position using the Jacobian pseudo-inverse update:

$$\Delta\boldsymbol{\phi} = J^{\dagger}\, \lambda\,(\hat{\mathbf{p}} - \mathbf{e})$$

The robot is limited to 8 IK iterations per step (`iteration_limit = 8`), which bounds per-step compute while still allowing meaningful end-effector movement. The end-effector repulsion zone ($r_{ee} = 2.0$, $k_{ee} = 100.0$) is deliberately large and strong to produce a clearly visible deflection effect on nearby agents.

### 10.4 Results

Weither or not the robot prevents any agents from escaping is depedent highly on the repulsion zone. From observations and experimentation done the threshold of about 2.5 is spotted, after 2.5 the end point will begin to trap agents in corners and stop them from reaching the exits.

In terms of interference the robot arm directly interfers but seems to have difficulty choosing what exit to priortize, this was mediated by making it recalculate clusters with longer intervals so that it sticks to one loctation longer.  

While it doesnt stop them from escaping we can see it visably slow down the particles, we can observe this most notably at the end with the 3 remaining particles begging at around 4 ish seconds.

Intrestingly enough the weight that the robot arm casts does not affect the particles greatly but does affect the prediction algorthim the arm uses far more, the larger the weight the more exagerated the predictions become causing the arm to jitter more violently and unpredictably.

---